# Amharic Sentiment Classifier — v3 (professionally-labeled Zenodo data)

**Source:** Girma Neshir Alemneh, "Negation handling for Amharic sentiment classification", Zenodo, DOI: 10.5281/zenodo.5005968 (CC-BY 4.0). Addis Ababa University research.

Two of the four files in this release are labeled by actual professionals:
- `ebc_facebook_news_only_Amharic_texts` — labeled by EBC (Ethiopian Broadcasting Corporation) professionals
- `news_reviews2009_2010_all` — labeled by GCAO professionals

Both are **binary** (positive/negative only — no neutral class), which matches what was flagged to me: published results on this kind of data report much higher accuracy than 3-class neutral-heavy tasks like AfriSenti, partly because binary sentiment is a genuinely easier task, not because our earlier work was wrong.

**Important — I have not been able to preview the raw file content directly** (Zenodo isn't reachable from where this notebook is generated). The filenames are unusually named (`.csvREM.xml`), so the cells below **download and print the raw content first** — look at what's printed before trusting the parsing cell after it, and paste me the Step 2 output if the parser doesn't work on the first try.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn torch

In [ ]:
import torch
import pandas as pd
import re
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import numpy as np

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

## Step 1: Download the two professionally-labeled files

In [ ]:
!wget -q "https://zenodo.org/records/5005968/files/ebc_facebook_news_only_Amharic_texts.csvREM.xml?download=1" -O ebc_facebook.raw
!wget -q "https://zenodo.org/records/5005968/files/news_reviews2009_2010_all.csvREM.xml?download=1" -O gcao_news.raw

!ls -lh ebc_facebook.raw gcao_news.raw

## Step 2: Print the raw content — LOOK AT THIS before running the next cell
This tells us the actual structure (real XML tags? Comma-separated? Tab-separated? Something else?) so we parse it correctly instead of guessing.

In [ ]:
with open("ebc_facebook.raw", "r", encoding="utf-8", errors="replace") as f:
    raw_preview = f.read(2000)  # first ~2000 characters

print(raw_preview)

## Step 3: Parse based on what you saw above
This cell tries the most likely format (real XML with tags) first, and falls back to CSV/TSV-style parsing if that fails. **Read the printed output from Step 2 before trusting this blindly** — if neither approach matches what you saw, paste me the Step 2 output and I'll write exact parsing code for it.

In [ ]:
import xml.etree.ElementTree as ET

def try_parse_file(path):
    """Attempts a few common formats. Returns a DataFrame with 'text' and 'label' columns, or None if all fail."""

    # Attempt 1: real XML
    try:
        tree = ET.parse(path)
        root = tree.getroot()
        rows = []
        for elem in root.iter():
            # Look for common tag name patterns — adjust if Step 2's output showed different tag names
            text_val = elem.findtext("text") or elem.findtext("Text") or elem.findtext("comment")
            label_val = elem.findtext("label") or elem.findtext("Label") or elem.findtext("sentiment")
            if text_val and label_val:
                rows.append({"text": text_val.strip(), "label": label_val.strip()})
        if rows:
            print(f"Parsed as XML: {len(rows)} rows found")
            return pd.DataFrame(rows)
    except ET.ParseError:
        pass

    # Attempt 2: tab-separated
    try:
        df = pd.read_csv(path, sep="\t", header=None, names=["text", "label"], on_bad_lines="skip", engine="python")
        if df["label"].nunique() <= 5:  # sanity check: labels should be a small set, not free text
            print(f"Parsed as tab-separated: {len(df)} rows")
            return df
    except Exception:
        pass

    # Attempt 3: comma-separated
    try:
        df = pd.read_csv(path, header=None, names=["text", "label"], on_bad_lines="skip", engine="python")
        if df["label"].nunique() <= 5:
            print(f"Parsed as comma-separated: {len(df)} rows")
            return df
    except Exception:
        pass

    return None

ebc_df = try_parse_file("ebc_facebook.raw")
gcao_df = try_parse_file("gcao_news.raw")

if ebc_df is not None:
    print("\nEBC sample:")
    print(ebc_df.head())
    print(ebc_df["label"].value_counts())
else:
    print("\nCould not auto-parse ebc_facebook.raw — paste the Step 2 output to Claude for a custom parser.")

if gcao_df is not None:
    print("\nGCAO sample:")
    print(gcao_df.head())
    print(gcao_df["label"].value_counts())
else:
    print("\nCould not auto-parse gcao_news.raw — paste the Step 2 output to Claude for a custom parser.")

## Step 4: Combine, clean, split
Only run this once both files above parsed successfully and the label counts look sane (2 unique values: positive/negative, in whatever form they take — e.g. `1`/`-1`, `pos`/`neg`, etc.).

In [ ]:
df = pd.concat([ebc_df, gcao_df], ignore_index=True)
print("Combined rows:", len(df))
print(df["label"].value_counts())

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)
    return text

df["text"] = df["text"].apply(clean_text)
df = df[df["text"].str.len() > 2].reset_index(drop=True)
print("After cleaning:", len(df))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print("Train:", len(train_df), "| Validation:", len(val_df), "| Test:", len(test_df))

## Step 5: Tokenize and fine-tune (binary this time)

In [ ]:
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_list = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: str(label) for label, i in label2id.items()}
print("Label mapping:", label2id)

def to_hf_dataset(pandas_df):
    pandas_df = pandas_df.copy()
    pandas_df["label"] = pandas_df["label"].map(label2id)
    return Dataset.from_pandas(pandas_df[["text", "label"]], preserve_index=False)

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

print(train_ds[0])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./amharic_sentiment_model_v3",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    warmup_steps=200,
    adam_epsilon=1e-6,
    adam_beta2=0.98,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## Step 6: Honest evaluation on the untouched test set

In [ ]:
test_predictions = trainer.predict(test_ds)
preds = np.argmax(test_predictions.predictions, axis=-1)
true_labels = test_predictions.label_ids

print("=== Test set performance ===")
print(classification_report(true_labels, preds, target_names=[id2label[i] for i in sorted(id2label)]))

cm = confusion_matrix(true_labels, preds)
print("Confusion matrix:")
print(cm)

## Step 7: Push to Hub (only once results look genuinely good)

In [ ]:
# Make sure your HF token has WRITE permission, and this matches your real username
model.push_to_hub("tys22/amharic-sentiment-xlmr-v3-binary")
tokenizer.push_to_hub("tys22/amharic-sentiment-xlmr-v3-binary")
print("Pushed to Hugging Face Hub.")